# Route Clustering Analysis

**Purpose:** assign GPS pings to mine zones, extract zone-to-zone haul routes (middling load -> reject dump), and cluster routes two ways: (1) KMeans on raw points for a quick spatial grouping, and (2) KMeans on resampled route-shape vectors, with per-route distance/duration/tortuosity/stop/noise metrics for comparison.

This is preliminary route-shape clustering (KMeans), distinct from the DBSCAN stop/idle-detection approach planned for the cycle-time paper — see `paper/draft_paper.md` for how the two relate.

In [ ]:
import sys
sys.path.append("../..")

import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gps_lib import io_utils, classify, preprocess, zones, routes, clustering, plotting

In [ ]:
tracker_list_df = io_utils.load_tracker_list()
tracker_list_df = classify.classify_technic_material_type(tracker_list_df)

track_points_df = io_utils.load_gps_data()
merged_df = preprocess.attach_technic_info(track_points_df, tracker_list_df)
df_cleaned = preprocess.clean_gps_points(merged_df, round_n=4)
df_cleaned.shape

In [ ]:
TARGET_DATE = datetime.date(2025, 11, 1)
gps = df_cleaned[
    (df_cleaned["date"] == TARGET_DATE)
    & (df_cleaned["technic_type"] == "dump")
    & (df_cleaned["technic_m_type"] == "other")
].copy()
gps.shape

In [ ]:
zone_list_df = io_utils.load_zone_list()
zone_list_df = classify.classify_zones(zone_list_df)
zone_detail_all_df = io_utils.load_zone_detail()

zones_gdf = zones.build_zone_geodataframe(zone_detail_all_df, zone_list_df)
zones_target = zones_gdf[zones_gdf["zone_material_type"].isin(["reject", "middling"])].copy()
zones_target[["zone_id", "label", "zone_material_type", "zone_load_type"]]

In [ ]:
# First-pass spatial clustering: raw points within the zones' bounding box
minx, miny, maxx, maxy = zones_target.total_bounds
margin = 0.002
gps_roi = gps[
    gps["lng"].between(minx - margin, maxx + margin)
    & gps["lat"].between(miny - margin, maxy + margin)
].copy()

K_POINTS = 6
coords = gps_roi[["lng", "lat"]].values
gps_roi["cluster"] = clustering.kmeans_cluster_points(coords, k=K_POINTS)
plotting.plot_point_clusters(zones_target, gps_roi, cluster_col="cluster")

In [ ]:
# Zone-hit assignment for the full cleaned dataset (needed for route extraction)
gps_hits = zones.assign_zone_hit(
    df_cleaned.sort_values(["tracker_id", "get_time"]).reset_index(drop=True),
    zones_target,
)
gps_hits[["tracker_id", "get_time", "lat", "lng", "zone_mat_hit", "zone_load_hit"]].head()

In [ ]:
route_list = routes.extract_routes(
    gps_hits,
    start_material="middling", start_load="load",
    end_material="reject", end_load="unload",
)
print("Extracted routes:", len(route_list))

In [ ]:
plotting.plot_routes_over_zones(zones_target, route_list)

In [ ]:
# Route-shape clustering: resample each route to a fixed-length vector, then KMeans
K_ROUTES = 6
route_vectors = np.vstack([routes.resample_route(r["points"], n=100) for r in route_list])
labels = clustering.kmeans_cluster_route_shapes(route_vectors, k=K_ROUTES)
for r, lbl in zip(route_list, labels):
    r["cluster"] = int(lbl)

plotting.plot_routes_over_zones(zones_target, route_list, color_by="cluster", k=K_ROUTES)

In [ ]:
small_reject_polys = zones_target.loc[zones_target["zone_material_type"] == "reject", "geometry"].tolist()
route_stats = routes.summarize_routes(route_list, small_reject_polys=small_reject_polys)
route_stats.head()

In [ ]:
plotting.plot_route_stat_boxplots(
    route_stats,
    metrics=["distance_m", "duration_sec", "tortuosity", "noise_index"],
)

In [ ]:
route_stats.groupby("cluster")[[
    "distance_m", "duration_sec", "tortuosity", "num_stops", "noise_index"
]].mean().round(2)

In [ ]:
sns.pairplot(
    route_stats,
    vars=["distance_m", "duration_sec", "tortuosity", "noise_index"],
    hue="cluster",
    palette="tab20",
    plot_kws=dict(alpha=0.6, s=40),
)
plt.show()